# MS COCO Hybrid KD: paired method comparison

Notebook CPU thực hiện paired bootstrap theo đúng `image_id` trên 5.000 ảnh
Karpathy Validation, so sánh Hybrid với Sequence KD và Dense Mapped Logit-KD.

## Input bắt buộc

1. Output `HYB_Audit`;
2. ba output Hybrid: `HYB_42`, `HYB_123`, `HYB_2026`;
3. ba output Sequence KD T1→S1 tương ứng seed `42`, `123`, `2026`;
4. ba output Dense Mapped Logit-KD T1→S1 tương ứng seed `42`, `123`, `2026`;
5. package/dataset `pycocoevalcap==1.2` offline nếu runtime chưa cài sẵn.

Runtime: **CPU / Accelerator None / Internet Off**. Dự kiến 15–30 phút.

In [1]:
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

EXPERIMENT_ID = "HYB_Compare"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 123, 2026)
REPLICATES = 10000
BOOTSTRAP_SEED = 20261101

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(temp, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    if len(matches) > 1:
        preferred = [
            path for path in matches
            if "output_v2" in {part.lower() for part in path.parts}
        ]
        if len(preferred) == 1:
            return preferred[0]
    assert len(matches) == 1, (name, [str(path) for path in matches])
    return matches[0]

audit_path = exactly_one("HYB_AUDIT.json")
audit = read_json(audit_path)
assert audit["status"] == "passed"
assert audit["decision"] == "AGGREGATE_PASSED_COMPARISON_REQUIRED"
assert audit["test_set_used"] is False

try:
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
except ImportError:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    assert wheels, "Thiếu pycocoevalcap hoặc offline wheel."
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheels[0])])
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

import pycocoevalcap.meteor.meteor as meteor_module

def fixed_meteor_init(self):
    source = Path(meteor_module.__file__).resolve().parent
    runtime = source
    if " " in str(source):
        runtime = OUTPUT_ROOT / "_meteor_runtime"
        shutil.copytree(source, runtime, dirs_exist_ok=True)
    self.meteor_cmd = [
        "java", "-Xmx2G", "-jar", str(runtime / "meteor-1.5.jar"),
        "-", "-", "-stdio", "-l", "en", "-norm",
    ]
    self.meteor_p = subprocess.Popen(
        self.meteor_cmd, cwd=runtime, stdin=subprocess.PIPE,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    self.lock = threading.Lock()

Meteor.__init__ = fixed_meteor_init

sequence_names = {
    42: "PHASE4_TEACHER_A_S1_SEQUENCE_KD_SEED42_FULL_RUN_COMPLETE.json",
    123: "PHASE4_TEACHER_A_S1_SEQUENCE_KD_SEED123_FULL_RUN_COMPLETE.json",
    2026: "PHASE4_TEACHER_A_S1_SEQUENCE_KD_SEED2026_FULL_RUN_COMPLETE.json",
}
dense_names = {
    42: "XLKD04_DENSE_MAPPED_LOGIT_KD_SEED42_FULL_RUN_COMPLETE.json",
    123: "XLKD05_DENSE_MAPPED_LOGIT_KD_SEED123_FULL_RUN_COMPLETE.json",
    2026: "XLKD05_DENSE_MAPPED_LOGIT_KD_SEED2026_FULL_RUN_COMPLETE.json",
}

def best_predictions(marker_path, marker):
    epoch = int(marker["best_result"]["epoch"])
    path = marker_path.parent / "validation" / f"epoch_{epoch:02d}_predictions.json"
    assert path.is_file(), path
    return path

sources = {}
for seed in SEEDS:
    hybrid_path = exactly_one(f"HYB_{seed}_COMPLETE.json")
    sequence_path = exactly_one(sequence_names[seed])
    dense_path = exactly_one(dense_names[seed])
    hybrid = read_json(hybrid_path)
    sequence = read_json(sequence_path)
    dense = read_json(dense_path)
    for marker in (hybrid, sequence, dense):
        assert marker["status"] == "complete"
        assert int(marker["seed"]) == seed
        assert marker.get(
            "selection_split",
            marker["best_result"].get("selection_split"),
        ) == "validation"
        assert marker["test_set_status"] == "sealed"
        assert marker["test_set_used"] is False
    references = hybrid_path.parent / "validation" / "karpathy_validation_references.json"
    assert references.is_file()
    reference_hash = sha256(references)
    for other_path in (sequence_path, dense_path):
        other_refs = other_path.parent / "validation" / "karpathy_validation_references.json"
        assert other_refs.is_file()
        assert sha256(other_refs) == reference_hash
    sources[seed] = {
        "references": references,
        "hybrid": (best_predictions(hybrid_path, hybrid), hybrid),
        "sequence": (best_predictions(sequence_path, sequence), sequence),
        "dense": (best_predictions(dense_path, dense), dense),
    }

def coco_payload(references, predictions, image_ids):
    raw_gts = {
        image_id: [{"caption": caption} for caption in references[str(image_id)]]
        for image_id in image_ids
    }
    raw_res = {
        image_id: [{"caption": predictions[str(image_id)]["caption"]}]
        for image_id in image_ids
    }
    tokenizer = PTBTokenizer()
    return tokenizer.tokenize(raw_gts), tokenizer.tokenize(raw_res)

def score_per_image(gts, res):
    output = {}
    _, bleu = Bleu(4).compute_score(gts, res)
    output["BLEU-1"] = np.asarray(bleu[0], dtype=np.float64)
    output["BLEU-4"] = np.asarray(bleu[3], dtype=np.float64)
    meteor = Meteor()
    try:
        _, values = meteor.compute_score(gts, res)
    finally:
        if hasattr(meteor, "close"):
            meteor.close()
    output["METEOR"] = np.asarray(values, dtype=np.float64)
    _, values = Rouge().compute_score(gts, res)
    output["ROUGE-L"] = np.asarray(values, dtype=np.float64)
    _, values = Cider().compute_score(gts, res)
    output["CIDEr-D"] = np.asarray(values, dtype=np.float64)
    return output

def paired_bootstrap(differences, rng):
    differences = np.asarray(differences, dtype=np.float64)
    values = np.empty(REPLICATES, dtype=np.float64)
    cursor = 0
    while cursor < REPLICATES:
        count = min(100, REPLICATES - cursor)
        indices = rng.integers(0, len(differences), size=(count, len(differences)))
        values[cursor:cursor + count] = differences[indices].mean(axis=1)
        cursor += count
    return {
        "mean_delta": float(differences.mean()),
        "lower_95": float(np.quantile(values, 0.025)),
        "upper_95": float(np.quantile(values, 0.975)),
        "replicates": REPLICATES,
    }

scores = {}
for seed in SEEDS:
    references = read_json(sources[seed]["references"])
    image_ids = sorted(map(int, references))
    assert len(image_ids) == 5000
    scores[seed] = {}
    for method in ("hybrid", "sequence", "dense"):
        prediction_path, marker = sources[seed][method]
        predictions = read_json(prediction_path)
        assert sorted(map(int, predictions)) == image_ids
        gts, res = coco_payload(references, predictions, image_ids)
        method_scores = score_per_image(gts, res)
        expected = float(marker["best_result"]["metrics"]["CIDEr-D"])
        observed = float(method_scores["CIDEr-D"].mean())
        assert np.isclose(observed, expected, atol=1e-6, rtol=0.0), (seed, method, observed, expected)
        scores[seed][method] = method_scores

comparisons = {}
csv_rows = []
for comparator in ("sequence", "dense"):
    key = f"hybrid_vs_{comparator}"
    comparisons[key] = {}
    for seed in SEEDS:
        comparisons[key][str(seed)] = {}
        rng = np.random.default_rng(BOOTSTRAP_SEED + seed + (0 if comparator == "sequence" else 10000))
        for metric in ("BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"):
            result = paired_bootstrap(scores[seed]["hybrid"][metric] - scores[seed][comparator][metric], rng)
            result["positive_lower_95"] = result["lower_95"] > 0.0
            comparisons[key][str(seed)][metric] = result
            csv_rows.append({"comparison": key, "seed": seed, "metric": metric, **result})

hybrid_beats_sequence = [
    seed for seed in SEEDS
    if comparisons["hybrid_vs_sequence"][str(seed)]["CIDEr-D"]["positive_lower_95"]
]
hybrid_beats_dense = [
    seed for seed in SEEDS
    if comparisons["hybrid_vs_dense"][str(seed)]["CIDEr-D"]["positive_lower_95"]
]
promote_hybrid = len(hybrid_beats_sequence) >= 2
decision = "PROMOTE_HYBRID" if promote_hybrid else "RETAIN_SEQUENCE_KD_REPORT_HYBRID_ABLATION"

result = {
    "schema_version": "hyb_compare_v1",
    "created_at_utc": now(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "unit": "validation image_id",
    "validation_images_per_seed": 5000,
    "bootstrap_replicates": REPLICATES,
    "comparisons": comparisons,
    "gates": {
        "hybrid_beats_sequence_positive_CIDEr_D_CI": f"{len(hybrid_beats_sequence)} / 3",
        "hybrid_beats_dense_positive_CIDEr_D_CI": f"{len(hybrid_beats_dense)} / 3",
        "required_for_promotion": "Hybrid > Sequence KD with positive lower 95% CI on at least 2 / 3 seeds",
    },
    "decision": decision,
    "held_out_test_may_be_opened_for_hybrid": False,
    "onnx_or_int8_authorized_for_hybrid": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "audit_sha256": sha256(audit_path),
}
result_path = OUTPUT_ROOT / "HYB_COMPARE.json"
write_json(result_path, result)

csv_path = OUTPUT_ROOT / "HYB_BOOTSTRAP_X100.csv"
with csv_path.open("w", encoding="utf-8-sig", newline="") as handle:
    fieldnames = list(csv_rows[0])
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for row in csv_rows:
        converted = dict(row)
        for name in ("mean_delta", "lower_95", "upper_95"):
            converted[name] = round(float(converted[name]) * 100, 4)
        writer.writerow(converted)

report = OUTPUT_ROOT / "REPORT.md"
report.write_text(
    "# MS COCO 2014 — Hybrid KD paired method comparison\n\n"
    f"- Hybrid thắng Sequence KD theo cổng CI dương: **{len(hybrid_beats_sequence)}/3 seed**.\n"
    f"- Hybrid thắng Dense Mapped Logit-KD theo cổng CI dương: **{len(hybrid_beats_dense)}/3 seed**.\n"
    f"- Quyết định: **{decision}**.\n"
    "- Tập Held-out Test không được đọc; Hybrid chưa được phép xuất ONNX/INT8.\n",
    encoding="utf-8",
)

artifacts = [result_path, csv_path, report]
write_json(OUTPUT_ROOT / "MANIFEST.json", {
    "schema_version": "hyb_compare_manifest_v1",
    "status": "complete",
    "artifacts": [
        {"relative_path": p.name, "size_bytes": p.stat().st_size, "sha256": sha256(p)}
        for p in artifacts
    ],
    "test_set_used": False,
})

for comparator in ("sequence", "dense"):
    print("Hybrid vs", comparator)
    for seed in SEEDS:
        cider = comparisons[f"hybrid_vs_{comparator}"][str(seed)]["CIDEr-D"]
        print(
            f"  seed={seed}: delta x100={cider['mean_delta'] * 100:+.2f}, "
            f"95% CI=[{cider['lower_95'] * 100:+.2f}, {cider['upper_95'] * 100:+.2f}]"
        )
print("Decision                  :", decision)
print("Held-out Test             : SEALED")

Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 307820 tokens at 588749.82 tokens per second.
PTBTokenizer tokenized 52130 tokens at 271446.40 tokens per second.


{'testlen': 47131, 'reflen': 47287, 'guess': [47131, 42131, 37131, 32131], 'correct': [35703, 19931, 9787, 4647]}
ratio: 0.9967009960454037


PTBTokenizer tokenized 307820 tokens at 599785.17 tokens per second.
PTBTokenizer tokenized 52177 tokens at 269578.52 tokens per second.


{'testlen': 47178, 'reflen': 47262, 'guess': [47178, 42178, 37178, 32178], 'correct': [35886, 20116, 9971, 4730]}
ratio: 0.998222673606682


PTBTokenizer tokenized 307820 tokens at 559439.23 tokens per second.
PTBTokenizer tokenized 52207 tokens at 245291.59 tokens per second.


... (9 outputs left out)

PTBTokenizer tokenized 307820 tokens at 599955.14 tokens per second.
PTBTokenizer tokenized 52120 tokens at 237247.94 tokens per second.


{'testlen': 47121, 'reflen': 47193, 'guess': [47121, 42121, 37121, 32121], 'correct': [35943, 20227, 9967, 4730]}
ratio: 0.9984743500095142


PTBTokenizer tokenized 307820 tokens at 544389.47 tokens per second.
PTBTokenizer tokenized 52190 tokens at 230439.32 tokens per second.


{'testlen': 47191, 'reflen': 47263, 'guess': [47191, 42191, 37191, 32191], 'correct': [35773, 19889, 9721, 4561]}
ratio: 0.9984766096100334
Hybrid vs sequence
  seed=42: delta x100=-1.65, 95% CI=[-2.98, -0.35]
  seed=123: delta x100=-1.94, 95% CI=[-3.21, -0.67]
  seed=2026: delta x100=-0.52, 95% CI=[-1.82, +0.78]
Hybrid vs dense
  seed=42: delta x100=+0.46, 95% CI=[-0.67, +1.61]
  seed=123: delta x100=+0.99, 95% CI=[-0.17, +2.15]
  seed=2026: delta x100=+2.17, 95% CI=[+1.10, +3.28]
Decision                  : RETAIN_SEQUENCE_KD_REPORT_HYBRID_ABLATION
Held-out Test             : SEALED


## Quy tắc quyết định

Hybrid chỉ được xem là ứng viên thay thế Sequence KD nếu chênh lệch CIDEr-D
có cận dưới khoảng tin cậy 95% lớn hơn 0 ở ít nhất 2/3 seed. Dù kết quả nào,
notebook này không mở Held-out Test.